summarization

# Middleware in LangChain

## What is Middleware?

**Middleware** is a layer that sits between the **user/application and the AI model**.

It allows us to **intercept, modify, validate, or control** the flow of information before or after the model runs.

```text
User Input
    ↓
Middleware
    ↓
LLM / Agent
    ↓
Middleware
    ↓
Final Response
```

## Why use Middleware?

Middleware is useful when we want to add common logic around an AI application without modifying the core agent/model logic.

### Common Use Cases

- Logging requests and responses
- Modifying prompts
- Validating inputs/outputs
- Adding authentication or permissions
- Handling errors
- Monitoring token usage
- Controlling tool calls
- Adding guardrails
- Filtering sensitive information
- Retry/fallback logic

## Middleware in LangChain

In **LangChain agents**, middleware can be used to control the agent's execution lifecycle.

It can run logic at different stages:

```text
Before Model
     ↓
   Model
     ↓
After Model
     ↓
 Tool Call
     ↓
After Tool
```

## Basic Example

```python
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    tools=tools,
    middleware=[
        ...
    ]
)
```

Middleware can inspect or modify what happens during the agent execution.

## Example Use Case

Suppose we want to prevent an agent from processing extremely long user inputs:

```text
User Input
    ↓
Middleware
    ↓
Check Input Length
    ↓
 ┌───────────────┐
 │ Valid Input?  │
 └───────┬───────┘
       Yes ↓    No → Reject
         LLM
```

This keeps the **agent logic clean** while middleware handles additional behavior.

## Key Idea

> **Middleware = A control layer around the AI workflow**

Instead of putting logging, validation, security, retries, etc. directly inside the agent logic, we can keep them separate using middleware.


In [11]:
import os
from dotenv import load_dotenv

load_dotenv()

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver


model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash"
)


agent = create_agent(
    model=model,

    checkpointer=InMemorySaver(),

    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("messages", 10),
            keep=("messages", 4)
        )
    ],

    system_prompt="You are a very sarcastic agent."
)

In [ ]:
##run with thread
config = {
    "configurable": {
        "thread_id": "test-1"
    }
}

questions = [
    "What is 2+2?",
    "What is 10*5?",
    "What is 100/4?",
    "What is 15-7?",
    "What is 3*3?",
    "What is 4*4?",
]

                create_agent()
                     │
                     ▼
              ┌─────────────┐
              │    Agent    │
              └──────┬──────┘
                     │
          ┌──────────┴──────────┐
          ▼                     ▼
   InMemorySaver       SummarizationMiddleware
          │                     │
          │                     │
    saves history        watches message count
          │                     │
          └──────────┬──────────┘
                     ▼
               thread_id
                 "test-1"
                     │
                     ▼
              SAME CONVERSATION
                     │
       ┌─────────────┼─────────────┐
       ▼             ▼             ▼
    Q1 → A1       Q2 → A2       Q3 → A3
                     ...
                     │
                     ▼
             history gets large
                     │
                     ▼
                summarize
                     │
                     ▼
          old context → summary
          recent context → kept